# Phase 2 — Target Engineering (RUL plafonnée à 125 cycles)

Objectif de cette phase : construire la vraie cible d'entraînement, la RUL (Remaining Useful Life),
à partir de `train_FD001.txt`, puis la plafonner à 125 cycles comme convenu dans le cahier des charges.

## 1. Imports et chargement des données

On reprend les mêmes colonnes et la même fonction de chargement que dans le notebook de la Phase 1
(`01_eda_preparation.ipynb`), pour rester cohérent avec le format du fichier brut NASA.

In [ ]:
import pandas as pd  # pour manipuler les donnees sous forme de tableau
import matplotlib.pyplot as plt  # pour la visualisation de controle

# Noms des 26 colonnes du fichier brut NASA (identique au notebook de la Phase 1)
COLUMNS = [
    "unit_number",    # identifiant du moteur
    "time_cycles",    # numero du cycle (= numero du vol) pour ce moteur
    "op_setting_1",   # reglage operationnel 1 (altitude)
    "op_setting_2",   # reglage operationnel 2 (nombre de Mach)
    "op_setting_3",   # reglage operationnel 3 (poussee)
]
for i in range(1, 22):  # on ajoute les 21 capteurs bruts, nommes sensor_1 a sensor_21
    # (ici on garde les 21 capteurs du fichier brut, y compris les 6 retires en Phase 1 :
    # ce notebook ne calcule que la RUL a partir de unit_number/time_cycles, il ne lit aucun capteur)
    COLUMNS.append(f"sensor_{i}")  # ici, on genere le nom de chaque capteur un par un


In [ ]:
DATA_DIR = "../data"  # chemin relatif depuis notebooks/ vers le dossier data

def load_fd001(split):
    """
    Charge le fichier train ou test de FD001 dans un DataFrame pandas.

    Parameters
    ----------
    split : str
        "train" ou "test" - selectionne quel fichier brut charger.

    Returns
    -------
    pandas.DataFrame
        Une ligne par (moteur, cycle), avec les 26 colonnes nommees.
    """
    path = f"{DATA_DIR}/{split}_FD001.txt"  # construit le chemin complet du fichier
    df = pd.read_csv(
        path,
        sep=r"\s+",     # le fichier brut est separe par des espaces (largeur variable), pas des virgules
        header=None,    # le fichier brut n'a pas de ligne d'en-tete
        names=COLUMNS,  # donc on fournit les noms de colonnes nous-memes
    )
    return df  # renvoie le tableau charge


In [ ]:
# Colonnes = schema du fichier brut (26 colonnes fixes, imposees par le format NASA) ;
# la Phase 2 ne lit aucun capteur, seulement unit_number et time_cycles (voir cellule precedente)
train = load_fd001("train")  # chargement du fichier d'entrainement FD001
train.shape  # verification rapide : on doit retrouver (20631, 26)


## 2. Calcul de la vraie RUL par cycle

Le fichier brut ne donne que le numero du cycle en cours (`time_cycles`), jamais directement
le nombre de cycles restants avant la panne. Mais dans `train_FD001.txt`, chaque moteur est suivi
**jusqu'a sa panne** : le dernier cycle enregistre pour un moteur correspond donc au moment de la panne.

La RUL a un cycle donne se calcule donc ainsi :

`RUL = dernier_cycle_du_moteur - cycle_actuel`

Exemple concret : si le moteur 1 a son dernier cycle enregistre a 200, alors au cycle 150,
sa RUL vaut 200 - 150 = 50 cycles restants.

In [ ]:
# Ici, on cree une colonne vide qui recevra la RUL calculee pour chaque ligne
train["RUL"] = 0  # valeur provisoire, remplacee ligne par ligne juste apres

# On parcourt chaque moteur un par un
for moteur_id in train["unit_number"].unique():  # unique() donne la liste des identifiants de moteurs
    masque = train["unit_number"] == moteur_id  # Vrai sur les lignes de ce moteur, Faux sinon

    # Le dernier cycle enregistre pour ce moteur correspond a sa panne
    dernier_cycle = train.loc[masque, "time_cycles"].max()  # valeur maximale de time_cycles pour ce moteur

    # La RUL de chaque ligne de ce moteur est la distance a ce dernier cycle
    train.loc[masque, "RUL"] = dernier_cycle - train.loc[masque, "time_cycles"]

train[["unit_number", "time_cycles", "RUL"]].head()  # verification rapide sur les premieres lignes


## 3. Vérification : la RUL doit décroître jusqu'à 0 à la panne

Pour le moteur 1, on regarde les toutes dernières lignes : la RUL doit valoir exactement 0
sur la toute dernière ligne (le cycle de la panne), et augmenter en remontant dans le temps.

In [ ]:
masque_moteur_1 = train["unit_number"] == 1  # on isole le moteur 1
train.loc[masque_moteur_1, ["time_cycles", "RUL"]].tail()  # les 5 dernieres lignes de ce moteur


## 4. Plafonnement de la RUL à 125 cycles (Piecewise Linear RUL)

En tout début de vie, les capteurs d'un moteur varient à peine d'un cycle à l'autre — le signal ne montre
presque aucun signe de dégradation. Pourtant, la RUL brute peut y valoir 300, 280, 250... Demander au modèle
de distinguer une RUL de 300 d'une RUL de 280 à partir d'un signal quasiment plat est une tâche impossible :
il n'y a simplement pas assez d'information dans les capteurs à ce stade pour le justifier.

La solution retenue par la quasi-totalité des projets de la communauté (et par notre cahier des charges) :
**plafonner la RUL à 125 cycles**. Concrètement : tant que la RUL brute dépasse 125, on la remplace par 125 ;
en dessous de 125, on garde la RUL brute inchangée. La courbe résultante est plate au début (à 125),
puis décroît linéairement jusqu'à 0 à la panne — d'où le nom "Piecewise Linear RUL" (RUL linéaire par morceaux).

In [ ]:
PLAFOND_RUL = 125  # plafond retenu, cf. cahier des charges section 3

# Ici, on cree la colonne RUL plafonnee, ligne par ligne
train["RUL_cappee"] = 0  # valeur provisoire

for index_ligne in train.index:  # on parcourt chaque ligne du tableau une par une
    rul_brute = train.loc[index_ligne, "RUL"]  # RUL brute de cette ligne

    if rul_brute > PLAFOND_RUL:  # si la RUL brute depasse le plafond
        train.loc[index_ligne, "RUL_cappee"] = PLAFOND_RUL  # on la remplace par le plafond
    else:  # sinon (RUL brute deja sous le plafond)
        train.loc[index_ligne, "RUL_cappee"] = rul_brute  # on la garde telle quelle

train[["unit_number", "time_cycles", "RUL", "RUL_cappee"]].head()  # verification rapide


### Preuve chiffrée : le signal est-il vraiment plus stable en début de vie ?

Un seul moteur ne prouve rien statistiquement — on refait le test sur 3 moteurs différents
(1, 50 et 100) plutôt que sur un seul, pour vérifier que le motif se reproduit.

Pour chaque moteur, on compare `sensor_11` sur la phase "début de vie" (RUL > 125) à la phase
"fin de vie" (RUL <= 125) : la moyenne et l'écart-type (la dispersion) des valeurs sur chaque phase.

In [ ]:
# Liste des 3 moteurs a tester : un pris au debut, un au milieu, un a la fin de la liste des identifiants
# (un seul moteur ne suffirait pas a prouver un motif general, il faut le voir se repeter)
moteurs_a_tester = [1, 50, 100]
sensor_a_verifier = "sensor_11"  # capteur choisi pour la preuve (Ps30, deja etudie en Phase 1)

# On parcourt les 3 moteurs un par un, avec une boucle explicite (pas de comprehension)
for moteur_id in moteurs_a_tester:
    masque_moteur = train["unit_number"] == moteur_id  # Vrai sur les lignes de ce moteur, Faux sinon

    # Phase "debut de vie" : tous les cycles ou il reste plus de 125 vols (RUL brute > 125)
    masque_debut = masque_moteur & (train["RUL"] > PLAFOND_RUL)  # combine les deux conditions (ce moteur ET debut de vie)
    valeurs_debut = train.loc[masque_debut, sensor_a_verifier]  # les valeurs du capteur sur cette phase seulement

    # Phase "fin de vie" : tous les cycles ou il reste 125 vols ou moins (RUL brute <= 125)
    masque_fin = masque_moteur & (train["RUL"] <= PLAFOND_RUL)
    valeurs_fin = train.loc[masque_fin, sensor_a_verifier]

    # mean() = moyenne des valeurs sur la phase ; std() = ecart-type, la dispersion autour de cette moyenne
    ecart_moyenne = valeurs_fin.mean() - valeurs_debut.mean()  # de combien la moyenne a bouge entre les 2 phases
    ratio_dispersion = valeurs_fin.std() / valeurs_debut.std()  # combien de fois plus disperse en fin de vie

    # Affichage des resultats pour ce moteur, avant de passer au suivant
    print(f"Moteur {moteur_id} :")
    print(f"  debut de vie -> moyenne={valeurs_debut.mean():.3f}, ecart-type={valeurs_debut.std():.3f}")
    print(f"  fin de vie   -> moyenne={valeurs_fin.mean():.3f}, ecart-type={valeurs_fin.std():.3f}")
    print(f"  ecart de moyenne = {ecart_moyenne:+.3f}   dispersion x{ratio_dispersion:.2f} en fin de vie")
    print()  # ligne vide pour separer visuellement chaque moteur dans la sortie


In [ ]:
# Version graphique du meme test que la cellule precedente : un sous-graphique par moteur,
# pour voir a l'oeil ce qu'on vient de mesurer en chiffres (moyenne/ecart-type)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)  # 3 graphiques cote a cote, meme echelle verticale
# sharey=True : les 3 graphiques partagent la meme echelle sur l'axe vertical, pour pouvoir les comparer directement

# on parcourt les 3 moteurs avec enumerate() pour avoir a la fois la position (i = 0, 1, 2)
# et l'identifiant du moteur (moteur_id) ; i sert a choisir dans quel sous-graphique dessiner
for i, moteur_id in enumerate(moteurs_a_tester):
    masque_moteur = train["unit_number"] == moteur_id  # Vrai sur les lignes de ce moteur

    # on trace la courbe du capteur brut, cycle par cycle, dans le sous-graphique numero i
    axes[i].plot(
        train.loc[masque_moteur, "time_cycles"],   # axe horizontal : le numero du cycle
        train.loc[masque_moteur, sensor_a_verifier],  # axe vertical : la valeur du capteur
    )

    # meme logique que dans la cellule precedente : on retrouve le premier cycle ou la RUL passe sous 125
    masque_sous_plafond = masque_moteur & (train["RUL"] <= PLAFOND_RUL)
    premier_cycle_sous_plafond = train.loc[masque_sous_plafond, "time_cycles"].min()

    # ligne verticale en pointilles a cet endroit precis, pour separer visuellement les 2 phases
    axes[i].axvline(premier_cycle_sous_plafond, linestyle="--", color="black")
    axes[i].set_title(f"Moteur {moteur_id}")  # titre au-dessus de chaque sous-graphique
    axes[i].set_xlabel("Cycle (vol)")  # legende de l'axe horizontal, repetee sur les 3

axes[0].set_ylabel(sensor_a_verifier)  # une seule etiquette verticale, a gauche, partagee par les 3 graphiques
fig.suptitle(f"{sensor_a_verifier} brut au fil des cycles - la ligne pointillee marque RUL = 125")  # titre general

plt.tight_layout()  # evite que les titres/legendes ne se chevauchent entre les 3 graphiques
plt.show()


## 5. Validation visuelle : RUL brute vs RUL plafonnée sur un moteur

C'est la validation attendue par le cahier des charges pour cette phase : une visualisation de la RUL brute
comparée à la RUL plafonnée sur un moteur, pour vérifier visuellement le plateau puis la décroissance linéaire.

In [ ]:
moteur_exemple = 1  # moteur choisi pour la visualisation finale de validation
masque_exemple = train["unit_number"] == moteur_exemple  # Vrai sur les lignes de ce moteur

fig, ax = plt.subplots(figsize=(9, 5))  # une seule figure, une seule paire d'axes

# 1ere courbe : la RUL brute (avant plafonnement), en pointilles pour bien la distinguer
ax.plot(
    train.loc[masque_exemple, "time_cycles"],  # axe horizontal : le cycle
    train.loc[masque_exemple, "RUL"],           # axe vertical : la RUL brute
    label="RUL brute",
    linestyle="--",
)
# 2eme courbe : la RUL plafonnee (apres application du plafond de 125), en trait plein et plus epais
ax.plot(
    train.loc[masque_exemple, "time_cycles"],
    train.loc[masque_exemple, "RUL_cappee"],
    label="RUL plafonnee a 125",
    linewidth=2,
)

ax.set_xlabel("Cycle (vol)")             # legende axe horizontal
ax.set_ylabel("RUL (cycles restants)")   # legende axe vertical
ax.set_title(f"Moteur {moteur_exemple} - RUL brute vs RUL plafonnee")  # titre du graphique
ax.legend()  # affiche la legende (RUL brute / RUL plafonnee) pour distinguer les 2 courbes

plt.tight_layout()  # evite que le titre/les legendes ne debordent de la figure
plt.show()
